<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="450">


<h1 style="font-family: Arial; color: navy; text-align: center; padding: 7px 0;">
Sesión 4 — Reservas de vida y escenarios de estrés
</h1>
<p style="font-family: Arial; color: navy; text-align: center; font-size: 16px; margin-top: -8px;">
WSP — Python aplicado a modelos actuariales
</p>
<hr style="border: 2px solid navy; width: 70%; margin: auto;">


<h1 style="background-color:#0070C0; color:white; text-align:center; font-family:Arial; padding:20px 0;">
🎯 Objetivos de la sesión
</h1>

Al terminar esta sesión vas a ser capaz de:

- **Calcular la Reserva de Riesgo en Curso (RRC = RPND)** de un producto de vida a partir de la base de cálculo y el porcentaje de prima no devengada.
- **Construir, paso a paso, el motor de reservas matemáticas de largo plazo**: tablas de mortalidad y caídas (qx/cx), triángulos de probabilidades, flujos nominal / probabilizado / financiero, y la reserva individual y agregada.
- **Aplicar ese mismo motor a escenarios de estrés**: mortalidad (+11%), caídas (±25%) y gastos administrativos (+10%), reconociendo qué triángulos hay que recalcular en cada escenario y cuáles se reutilizan.
- **Construir una matriz de escenarios y una matriz de riesgos** que resuma el impacto de cada estrés frente al escenario base — el insumo directo de un informe de riesgo de reservas.
- **Reconocer dónde encaja el factor de nivelación** (ver anexo `anexos/anexo_factor_nivelacion.ipynb`) dentro de este flujo, sin tener que resolverlo en esta sesión.


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
✍️ Cómo usar este notebook
</h1>

Este notebook combina teoría, ejemplos resueltos y ejercicios guiados, organizados en tres partes:

- **Parte A** — Reserva de Riesgo en Curso (RRC).
- **Parte B** — Reservas matemáticas de largo plazo: aquí se construye, **una sola vez**, el "motor" de tablas y triángulos (qx, cx) que se reutiliza en el resto del notebook.
- **Parte C** — Escenarios de estrés (mortalidad, caídas, gastos) y reportería de riesgos, construidos **sobre el motor de la Parte B** (no se vuelve a cargar ni limpiar la base).
- **Ejercicios de cierre** — dos ejercicios adicionales sobre el motor de reservas matemáticas (caídas como decremento y una cobertura complementaria).

Donde encuentres `***` hay un hueco por completar: puede ser un nombre de columna, un operador, un número o una palabra. El código que lo rodea ya está resuelto — tu trabajo es descifrar qué falta para que la celda haga lo que dice el comentario de arriba. Ejecuta las celdas en orden: la Parte B construye variables (`rmat`, `qx`, `cx`, `aux_vigencia`, `beneficio`, `gastos`, `prima`, `mat_factor_descuento`...) que la Parte C y los ejercicios de cierre reutilizan directamente.

Si te atoras, el solucionario (`solucionarios/sesion_04_reservas_vida_solucionario.ipynb`) tiene exactamente la misma estructura, celda a celda, solo con los `***` resueltos. El detalle del **factor de nivelación** (siniestros esperados, factores por género/año y shocks de mortalidad/longevidad) vive aparte, en `anexos/anexo_factor_nivelacion.ipynb`.


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Librerías
</h1>

<p style = "font-family: Arial; color:black;"> En este bloque se importan las librerías de python que utilizamos para el cálculo de reserva. Estas librerías nos permiten trabajar con diferentes estructuras de datos, realizar distintas operaciones entre ellas y graficar.
</div>

### ⚙️ Celda de arranque

Si trabajas en **Google Colab**, ejecuta esta celda **antes que cualquier otra**: descarga los datos del curso desde GitHub y deja el notebook listo para leerlos. Vuelve a ejecutarla cada vez que Colab reinicie el entorno. En tu PC (instalación local) no hace nada.

In [ ]:
# ⚙️ Celda de arranque: prepara el entorno en Google Colab (en tu PC no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    %cd /content/python_actuarios/notebooks

In [ ]:
import numpy as np
import pandas as pd
import datetime


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Parámetros
</h1>

In [ ]:
# Parámetro principal: fecha de corte para el cálculo de reservas (fecha de valoración).
# Cambiar este valor para recalcular las reservas a otra fecha.
fecha_calculo = datetime.datetime.strptime(***, '%Y-%m-%d')


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Rutas de datos
</h1>

In [ ]:
from pathlib import Path

# Carpeta de datos: funciona tanto si el notebook corre desde "materiales_v2" (usa "datos")
# como desde una subcarpeta como "_reparados" (usa "../datos")
DATOS = Path(***) if Path("../datos").exists() else Path("datos")


---
# **CÁLCULO DE RESERVA DE RIESGO EN CURSO**
---

<div style="border: 1px solid #FFFFFF; padding: 20px; background-color: #DAE3F3;">
<h3>🧮 Cálculo de la RRC (Reserva de Riesgo en Curso) – Producto de Vida</h3>

$$
\mathrm{RRC} = \mathrm{RPND}
$$
<hr>


<h4>📘 Cálculo de la RPND (Reserva para Primas No Devengadas)</h4>

$$
\mathrm{RPND} = \text{Base de cálculo} \times \mathrm{pnc}
$$

<p>
- <b>Base de Cálculo</b>: la <b>prima neta</b> (prima emitida menos devoluciones y anulaciones)
</p>

<p>
- El <b>porcentaje de prima no devengada (pnc): </b> la proporción del tiempo de cobertura pendiente:</p>

$$
\mathrm{pnc} = \frac{\mathrm{FF} - \max(\mathrm{FC}, \mathrm{FI})}{\mathrm{FF} - \mathrm{FI}}
$$

<p>Donde:</p>
<ul>
  <li><b>FF:</b> Fecha de Fin de vigencia</li>
  <li><b>FC:</b> Fecha de Cálculo</li>
  <li><b>FI:</b> Fecha de Inicio de vigencia</li>
</ul>

<p>
<b>Nota:</b> el <code>max(FC, FI)</code> se implementa calculando una <code>fc_efectiva</code> (máximo entre la fecha de cálculo y la fecha de inicio); además, el código acota el resultado con <code>.clip(lower=0)</code> para las pólizas que ya vencieron (FF &lt; FC), de modo que <code>pnc</code> siempre quede entre 0 y 1.
</p>

</div>

1️⃣ Importando la base de datos

In [ ]:
rrc_ori = pd.read_excel(DATOS / ***)
rrc = rrc_ori.copy()


In [ ]:
rrc.dtypes

2️⃣Calculo de variables (en años)

In [ ]:
# FC efectiva: si la póliza aún no inicia (FC < FI), se usa FI como referencia (ver fórmula: max(FC, FI))
fc_efectiva = rrc['fecha inicio'].clip(lower=pd.Timestamp(***))
rrc['FF-FC'] = ((rrc['fecha fin'] - fc_efectiva).dt.days / ***).round(4)
rrc['vigencia'] = ((rrc['fecha fin'] - rrc['fecha inicio']).dt.days / 365.25).round(4)


3️⃣Calculo de la base de calculo

In [ ]:
rrc['gastos_adq'].describe()

In [ ]:
rrc['Base de Cálculo'] = rrc['monto prima'] * (1 - rrc[***])


In [ ]:
# Conversión de primas a soles (PEN) usando el tipo de cambio y la columna 'moneda' de la base
# (la base de ejemplo viene 100% en PEN, pero el cálculo generaliza al caso con primas en USD)
tc = ***  # tipo de cambio referencial USD/PEN a la fecha de cálculo

rrc['monto prima PEN'] = np.where(rrc['moneda'] == ***, rrc['monto prima'] * tc, rrc['monto prima'])

rrc['Base de Cálculo'] = rrc['monto prima PEN'] * (1 - rrc['gastos_adq'])


4️⃣Calculo de la RRC = RPND

In [ ]:
# Calculando la pnc = proporción de tiempo de cobertura pendiente (acotada entre 0 y 1)
rrc['pnc'] = rrc['FF-FC'] / rrc[***]
rrc['pnc'] = rrc['pnc'].clip(lower=***)  # cubre pólizas ya vencidas (FF < FC efectiva)

# Calculando la RPND
rrc['RPND'] = rrc['Base de Cálculo'] * rrc['pnc']

# Cuadro resumen
resumen_rrc = rrc.groupby(['nombre_producto', 'moneda']).agg({'RPND':***})
resumen_rrc = resumen_rrc.reset_index()
resumen_rrc.loc[len(resumen_rrc)] = ['Total', '', resumen_rrc['RPND'].sum()]
pd.options.display.float_format = '{:,.2f}'.format
resumen_rrc


---
# **CÁLCULO DE RESERVAS MATEMÁTICAS DE LARGO PLAZO**
# *Modelación propia*
---

![Esquema de flujos: parte nominal y parte probabilística](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_flujos1y2.jpg)


![Esquema de flujos: parte financiera](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_flujo3.jpg)


1️⃣ Importar la base de datos

➡️ 1. Ruta de los archivos

In [ ]:
ruta_rmat = DATOS / ***


In [ ]:
ruta_tabla = DATOS / ***


In [ ]:
ruta_vtd = DATOS / ***


➡️ 1. Importar archivos

In [ ]:
rmat_bruto = pd.read_csv(ruta_rmat)
tabla_qx = pd.read_excel(***)
vtd = pd.read_excel(ruta_vtd)


🧹 Limpieza mínima de la base de pólizas (la limpieza completa se realizó en la sesión de calidad de datos)

In [ ]:
# Limpieza mínima de la base "sucia" (la limpieza completa se vio en la sesión de calidad de datos):
# se descartan fechas nulas, montos nulos/negativos y pólizas duplicadas por num_poliza
rmat_ori = rmat_bruto.dropna(subset=['fecha nacimiento', 'fecha inicio', 'fecha fin', 'monto prima', 'suma_asegurada'])
rmat_ori = rmat_ori[(rmat_ori['monto prima'] > 0) & (rmat_ori[***] > 0)]
rmat_ori = rmat_ori.drop_duplicates(subset=***, keep='first').reset_index(drop=True)


2️⃣ Tablas de mortalidad

In [ ]:
tabla_qx

In [ ]:
tabla_MP = tabla_qx[[***, 'qx']]
pd.options.display.float_format = '{:,.7f}'.format
tabla_MP.head()


🛠️ **Construcción de una matriz de probabilidades**

![Construcción de la matriz de probabilidades: pasos 1 a 3](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_tabla_probabilidades.jpg)


In [ ]:
tabla_ejemplo = pd.DataFrame({'qx': [0.02, 0.03, 0.05, 0.07]})
pd.options.display.float_format = '{:,.2f}'.format
tabla_ejemplo


 ⚙️ 1. índices del vector con forma de matriz

In [ ]:
#tabla = tabla_ejemplo.copy()
tabla = ***.copy()
n = tabla.shape[0]


In [ ]:
#Auxiliar AP:
c = np.array((range(0,n)))[***]

def aux_AP(c): return np.array(range(0,c+1))
AP = list(map(aux_AP,c))
AP = np.concatenate(AP)
AP


In [ ]:
#Auxiliar DP:
DP  = np.repeat(range(0,n+1),(range(0,n+1)[***]))
DP


In [ ]:
[[AP, DP]]

In [ ]:
[0, 1, 2, 3, 1, 2, 3, 2, 3, 3]

In [ ]:
#Auxiliar DPc
CY = AP + ***
CY


 ⚙️ 2. Flujos a partir de un vector

In [ ]:
qx_vector = tabla.qx[***].to_numpy()
qx_vector


In [ ]:
n = tabla.shape[0]
triangle_m = np.zeros((***, n))
triangle_m


In [ ]:
f = 0
for col in range(0,n):
    for fila in range(0,n):
        triangle_m[fila][col] = qx_vector[f]
        f = f+1
    n=n-1

triangle_m = pd.DataFrame(triangle_m)
triangle_m.insert(0, ***, np.arange(0, triangle_m.shape[0]))
triangle_m


🔻 **Aplicando la misma técnica a las caídas (cx)**

La tabla `tabla_qx` también trae una columna `cx` (probabilidad de caída/lapso). Usamos el mismo vector de índices `CY` (construido arriba para `qx`) para armar el triángulo `triangle_c` — así el motor queda listo con **ambas** tablas de decremento (mortalidad y caídas) antes de pasar a las pólizas.


In [ ]:
tabla2 = tabla_qx[[***, 'cx']]
tabla2.head()


In [ ]:
cx_vector = tabla2.cx[***].to_numpy()


In [ ]:
n = tabla2.shape[0]
triangle_c = np.zeros((n, ***))

f = 0
for col in range(0,n):
    for fila in range(0,n):
        triangle_c[fila][col] = cx_vector[f]
        f = f+1
    n=n-1

triangle_c = pd.DataFrame(triangle_c)
triangle_c.insert(0, ***, np.arange(0, triangle_c.shape[0]))
triangle_c


3️⃣ Información de las pólizas

In [ ]:
rmat_ejemplo = pd.DataFrame({
    'num_poliza': ['c-1000009000001', 'c-1000009000002'],
    'nombre_producto': ['producto01', 'producto03'],
    'riesgo': ['Vida Grupo Particular', 'Vida Grupo Particular'],
    'sexo': ['F', 'M'],
    'moneda': ['PEN', 'PEN'],
    'fecha nacimiento': ['1992-05-14', '1988-11-03'],
    'fecha inicio': ['2020-01-01', '2023-01-01'],
    'fecha fin': ['2030-10-24', '2032-10-24'],
    'gastos_adm': [0.09, 0.09],
    'monto prima': [50.00, 75.00],
    'suma_asegurada': [15000.00, 22000.00]})

rmat_ejemplo


⚙️1. Cálculo de variables

In [ ]:
# Paso pedagógico ya revisado arriba: rmat_ejemplo (2 pólizas de juguete) sirve para validar la lógica.
# Para el cálculo real usamos la base completa ya limpiada (rmat_ori).
rmat = ***.copy()
# rmat = rmat_ejemplo.copy()  # <- alternativa: correr todo el notebook solo con el ejemplo de 2 pólizas


In [ ]:
rmat['vigencia'] = np.ceil((pd.to_datetime(rmat['fecha fin']) - pd.to_datetime(rmat['fecha inicio'])).dt.days / ***).astype(int)

rmat['t_poliza'] = np.ceil((pd.to_datetime(fecha_calculo) - pd.to_datetime(rmat['fecha inicio'])).dt.days / 365.25).astype(int)

rmat['Edad_actuarial'] = (fecha_calculo.year - pd.DatetimeIndex(rmat['fecha nacimiento']).year
                          + (fecha_calculo.month - pd.DatetimeIndex(rmat['fecha nacimiento']).month) / 12
                          + (fecha_calculo.day - pd.DatetimeIndex(rmat['fecha nacimiento']).day) / 365.25).astype(int)

rmat['periodo restante'] = (rmat['vigencia'] - rmat['t_poliza']).astype(int) + ***

rmat['monto gasto'] = rmat['monto prima'] * rmat[***]


# Campo sintético 'dotal': se usa más adelante en la Parte C (riesgo de mortalidad/caídas/gastos)
# para modelar un beneficio adicional que se paga si la póliza llega viva al fin de la vigencia.
np.random.seed(7)
rmat['dotal'] = np.where(np.random.rand(len(rmat)) < 0.3, rmat['monto prima'] * np.random.uniform(0.5, 1.5, len(rmat)), 0)


In [ ]:
rmat['gastos_adm'].describe()

⚙️2. Matriz auxiliar del período restante de vigencia

In [ ]:
#-------------------------------------------------------------------------------------------------------------------------------------------------------
# Creación de la matriz
#-------------------------------------------------------------------------------------------------------------------------------------------------------
matriz_polizas = np.zeros((rmat.shape[0], max(rmat[***])))
matriz_polizas[:,0] = rmat['periodo restante']
ncp = matriz_polizas.shape[1]

#-------------------------------------------------------------------------------------------------------------------------------------------------------
#Según edad actuarial:
#-------------------------------------------------------------------------------------------------------------------------------------------------------
def repet (matriz_polizas): return np.concatenate((np.repeat(1,matriz_polizas[0]),
                                                   np.repeat(0,ncp-matriz_polizas[0])),axis=0)
mat_zer = pd.DataFrame(list(map(repet,matriz_polizas)))
mat_zer.insert(0,'Edad',rmat['Edad_actuarial'])
mat_zer.insert(1, 't_res', rmat['periodo restante'])
mat_zer.insert(2, 'num_poliza', ***)


⚙️3. Construcción de una matriz de probabilidades para cada asegurado

🔸🔧 1. Unión de dos bases de datos a través de una llave

In [ ]:
qx_por_edad = triangle_m.iloc[:,0:(max(rmat['periodo restante'])+1)]
aux_qx = pd.merge(mat_zer, qx_por_edad, on=***, how='left')


In [ ]:
pd.set_option('display.max_columns', None)
aux_qx


In [ ]:
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')


In [ ]:
#UNION rmatS: Se trabaja por separado cada una de las rmats unidas:
qx1 = (aux_qx.iloc[:,3:mat_zer.shape[1]])
qx1.rename(columns=lambda x: x.replace('_x', ''), inplace=True)
qx2 = (aux_qx.iloc[:,(mat_zer.shape[1]):aux_qx.shape[1]])
qx2.rename(columns=lambda x: x.replace(***, ''), inplace=True)

qx = pd.concat([aux_qx['num_poliza'],qx1 * qx2], axis=1)
qx.columns = ['certificado'] + list(np.arange(0, qx.shape[1]-1))

pd.options.display.float_format = '{:,.7f}'.format
qx.head()


🔸🔧 2. Unión de dos bases de datos a través de una llave (cx)

Igual que hicimos con `qx` por `Edad`, ahora cruzamos `triangle_c` con las pólizas por `t_poliza` (antigüedad), usando una matriz auxiliar `mat_zer2` equivalente a `mat_zer` pero indexada por `t`.


In [ ]:
mat_zer2 = mat_zer.iloc[:, 3:mat_zer.shape[1]]
mat_zer2.insert(0,'t', rmat[***])
mat_zer2.insert(1, 'num_poliza', rmat['num_poliza'])
mat_zer2.head()


In [ ]:
caidas = triangle_c.iloc[:,0:(max(rmat['periodo restante'])+1)]

aux_cx = pd.merge(mat_zer2, caidas, on=***, how='left')

#UNION: Se trabaja por separado cada una de las matrices unidas:
cx1 = (aux_cx.iloc[:,2:mat_zer2.shape[1]])
cx1.rename(columns=lambda x: x.replace('_x', ''), inplace=True)
cx2 = (aux_cx.iloc[:,(mat_zer2.shape[1]):aux_cx.shape[1]])
cx2.rename(columns=lambda x: x.replace('_y', ''), inplace=True)

cx = pd.concat([aux_cx['num_poliza'],cx1 * cx2], axis=1)
cx.columns = ['num_poliza'] + list(np.arange(0, cx.shape[1]-1))

pd.options.display.float_format = '{:,.7f}'.format
cx.head()


4️⃣Construcción de flujos

Ⓐ Parte Nominal

![Esquema de cálculo: construcción de la matriz de beneficios](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_esquema_polizas.jpg)


![Esquema de cálculo: parte nominal (beneficios, primas, gastos)](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_parte_nominal.jpg)


In [ ]:
aux_vigencia = mat_zer.iloc[:, ***:mat_zer.shape[1]]
aux_vigencia.head()


📉 Beneficio (Suma asegurada)

In [ ]:
beneficio = pd.concat([rmat[***]] * max(rmat['periodo restante']), axis=1, ignore_index=True)
beneficio = beneficio * aux_vigencia
beneficio.head()


📉 Primas

In [ ]:
prima = pd.concat([rmat['monto prima']] * max(rmat[***]), axis=1, ignore_index=True) * aux_vigencia
pd.options.display.float_format = '{:,.2f}'.format
prima.head()


📉 Gastos administrativos

In [ ]:
gastos = pd.concat([rmat['monto gasto']] * max(rmat['periodo restante']), axis=1, ignore_index=True) * ***
pd.options.display.float_format = '{:,.2f}'.format
gastos.head()


Ⓑ Parte probabilizada

![Esquema de cálculo: parte probabilizada](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_parte_probabilizada.jpg)


- **px:** probabilidad de sobrevivir hasta la edad x+1

- **qx:** Probabilidad de que una persona de edad x muera antes de 1 año.


🌀flujo de sobrevivencia (px)

In [ ]:
qx_matriz = qx.iloc[:,***:-1]
qx_matriz


In [ ]:
px =  (*** - qx_matriz)


In [ ]:
triangle_f = px.copy()
triangle_f = triangle_f.cumprod(axis=***)
triangle_f = pd.DataFrame(triangle_f)


In [ ]:
triangle_f.insert(0,'sobrevivencia',***)
cols = np.arange(0,triangle_f.shape[1])
triangle_f.columns = cols

triangle_f = triangle_f * aux_vigencia

pd.options.display.float_format = '{:,.7f}'.format
triangle_f.head()


🌀flujo de mortalidad (qx)

In [ ]:
qx.iloc[:,1:qx.shape[1]]

In [ ]:
triangulo_cobertura = qx.iloc[:,1:qx.shape[1]] * triangle_f.iloc[:,0:***]


In [ ]:
triangulo_cobertura.insert(0,'cobertura1',***)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_cobertura.columns = cols

triangulo_cobertura = triangulo_cobertura.iloc[:, :-1] * mat_zer.iloc[:,3:mat_zer.shape[1]]
triangulo_cobertura


In [ ]:
triangulo_cobertura.insert(0,'num_poliza', rmat[***])
triangulo_cobertura = triangulo_cobertura.iloc[:,0:(max(rmat['periodo restante'])+1)]
triangulo_cobertura.head()


Ⓒ Parte financiera

![Esquema de cálculo: parte financiera](https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/fig_parte_financiera.jpg)


In [ ]:
#----------------------------------------------------------------------------------------------------------------------
# FACTOR DE DESCUENTO
#----------------------------------------------------------------------------------------------------------------------
# El vector de tasas viene con granularidad MENSUAL (columna 'plazo_meses'); el modelo trabaja en
# pasos ANUALES, así que tomamos la tasa en cada aniversario (plazo_meses = t * 12)
n_periodos = max(rmat['periodo restante'])
t = pd.DataFrame(np.arange(0, n_periodos))

vector_descuento = pd.DataFrame(vtd.set_index('plazo_meses')['tasa'].reindex(t[0] * ***).reset_index(drop=True))

tasa = pd.DataFrame(np.concatenate([t,vector_descuento],axis=1))
factor_descuento = np.divide(1, pow(1+(tasa[1]),tasa[***]))
pd.options.display.float_format = '{:,.5f}'.format
factor_descuento.head(3)


In [ ]:
mat_factor_descuento = np.transpose(pd.DataFrame(factor_descuento))
mat_factor_descuento = mat_factor_descuento.loc[mat_factor_descuento.index.repeat(rmat.shape[0])].reset_index(drop=***) * aux_vigencia
mat_factor_descuento.head()


5️⃣Calculo de reserva

In [ ]:
flujo_cob1 = beneficio * triangulo_cobertura.iloc[:, ***:]
flujo_gastos = gastos * triangle_f
flujo_primas = prima * triangle_f
flujo_primas


In [ ]:
egresos =  flujo_cob1 + flujo_gastos
ingresos = flujo_primas
reserva = (*** - ingresos) * mat_factor_descuento
reserva.head()


In [ ]:
reserva_individual = reserva.sum(axis=***)
reserva_individual = pd.DataFrame(reserva_individual)
reserva_individual[reserva_individual < 0] = 0
reserva_individual.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individual.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individual.head()


6️⃣Reportería

In [ ]:
resumen = reserva_individual.copy()
resumen['nombre_producto'] = rmat['nombre_producto']
cuadro_resumen = resumen.groupby([***]).agg({'Reservas':'sum'})

pd.options.display.float_format = '{:,.2f}'.format
cuadro_resumen


In [ ]:
(flujo_cob1 * mat_factor_descuento)

---
# **PARTE C — ESCENARIOS DE ESTRÉS**
---


A partir de aquí reutilizamos **todo lo construido en la Parte B**: `rmat`, `mat_zer`, `mat_zer2`, `aux_vigencia`, `qx`, `cx`, `beneficio`, `gastos`, `prima` y `mat_factor_descuento`. No se vuelve a importar ni a limpiar la base de pólizas.

La diferencia frente a la Parte B es que aquí el escenario **base** ya incorpora la probabilidad de caída (además de la de mortalidad) en el flujo de sobrevivencia, y se agrega un flujo adicional: **dotal** (un capital que se paga si la póliza llega viva al fin de la vigencia). Sobre ese escenario base se aplican los estreses de mortalidad, caídas y gastos, y se arma una matriz de escenarios y de riesgos.


5️⃣Construcción de flujos

Ⓐ Parte Nominal

`aux_vigencia`, `beneficio`, `gastos` y `prima` ya están construidos en la Parte B. Aquí solo agregamos el flujo de **dotal**.


In [ ]:
#-------------------------------------------------------------------------------------------------
# Dotal (usa aux_vigencia, mat_zer2 y rmat['dotal'], ya construidos en la Parte B)
#-------------------------------------------------------------------------------------------------
per_restante = np.asarray(rmat['periodo restante'])

matD = np.arange(max(rmat['periodo restante'])) > np.array(per_restante-***)[:, None]  # ¿en qué columna (índice) cae el último período de vigencia?
mat_zerD = pd.DataFrame(matD.astype(int))
mat_zerD.insert(0, 'Vo', rmat['periodo restante'])
mat_zerD.insert(1, 'num_poliza', rmat['num_poliza'])
mat_zerD.loc[mat_zerD['Vo'] == 0, np.arange(0,mat_zerD.shape[1]-3)] = 0

auxDotal = ((mat_zer2.iloc[:,2:(mat_zer2.shape[1])+1]) * mat_zerD.iloc[:,2:mat_zerD.shape[1]+2])

dotal = pd.concat([rmat['dotal']] * max(rmat['periodo restante']), axis=1, ignore_index=True) * auxDotal


Ⓑ Parte probabilizada

<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/flujo_prob.jpg" width="750">


In [ ]:
#-------------------------------------------------------------------------------------------------
# 🌀flujo de sobrevivencia (px) — ahora con mortalidad Y caídas
#-------------------------------------------------------------------------------------------------
mortalidad = qx.iloc[:,1:-1]
caidas = cx.iloc[:,1:cx.shape[1]-1]
px =  (1 - mortalidad) * (*** - caidas)  # ¿con qué complementamos "caidas" para pasar de probabilidad a permanencia?

triangle_f = px.copy()
triangle_f = triangle_f.cumprod(axis=1)
triangle_f = pd.DataFrame(triangle_f)

triangle_f.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_f.shape[1])
triangle_f.columns = cols

triangle_f = triangle_f * aux_vigencia

#-------------------------------------------------------------------------------------------------
# 🌀flujo de mortalidad (qx)
#-------------------------------------------------------------------------------------------------
triangulo_cobertura = qx.iloc[:,1:] * triangle_f

triangulo_cobertura.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_cobertura.columns = cols

triangulo_cobertura = triangulo_cobertura.iloc[:, :-1] * mat_zer.iloc[:,3:]
triangulo_cobertura.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_cobertura = triangulo_cobertura.iloc[:,0:(max(rmat['periodo restante'])+1)]

#-------------------------------------------------------------------------------------------------
# 🌀flujo de caidas (cx)
#-------------------------------------------------------------------------------------------------
triangulo_caidas = cx.iloc[:,1:cx.shape[1]] * ***  # ¿con qué triángulo de sobrevivencia se pondera la salida por caída?

triangulo_caidas.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidas.columns = cols

triangulo_caidas = triangulo_caidas.iloc[:, :-1] * mat_zer2.iloc[:,***]  # misma columna de corte que usamos para aux_vigencia (mat_zer)
triangulo_caidas.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidas = triangulo_caidas.iloc[:,0:(max(rmat['periodo restante'])+1)]


Ⓒ Parte financiera

In [ ]:
t = pd.DataFrame(np.arange(0,max(rmat['periodo restante'])))

vector_descuento = pd.DataFrame(vtd['tasa'])
vector_descuento = vector_descuento[0:len(t)]

tasa = pd.DataFrame(np.concatenate([t,vector_descuento],axis=1))
factor_descuento = np.divide(1, pow(1+(tasa[1]),tasa[0]))

mat_factor_descuento = np.transpose(pd.DataFrame(factor_descuento))
mat_factor_descuento = mat_factor_descuento.loc[mat_factor_descuento.index.repeat(rmat.shape[0])].reset_index(drop=True) * aux_vigencia

6️⃣Cálculo de reserva

In [ ]:
flujo_cob = beneficio * triangulo_cobertura.iloc[:, 1:]
flujo_gastos = gastos * triangle_f
flujo_dotal = dotal * ***  # ¿con qué triángulo se paga el dotal? (sobrevivencia, no caídas)
flujo_primas = prima * triangle_f

egresos =  flujo_cob + flujo_gastos + flujo_dotal
ingresos = flujo_primas

reserva = (egresos - ingresos) * mat_factor_descuento
sumReserva = reserva.sum(axis=1)


7️⃣Reserva individual

In [ ]:
reserva_individual = reserva.sum(axis=1)
reserva_individual = pd.DataFrame(reserva_individual)
reserva_individual[reserva_individual < 0] = 0
reserva_individual.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individual.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individual.head()

# ☠️ Riesgo de mortalidad
- **Estrés de 11%**

In [ ]:
mat_mortalidad = qx.iloc[:,1:]
mat_mortalidadEM = mat_mortalidad * ***  # estrés de mortalidad: +11%


🔔❗ Nuevo flujo de sobrevivencia (Px)

In [ ]:
mat_caidas = cx.iloc[:,1:]
pxEM =  (1 - mat_mortalidadEM.iloc[:,:-1]) * (1-***.iloc[:,:-1])  # ¿la caída se estresa en este escenario, o se queda en su valor base?

triangle_fEM = pxEM.copy()
triangle_fEM = triangle_fEM.cumprod(axis=1)
triangle_fEM = pd.DataFrame(triangle_fEM)
triangle_fEM.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_fEM.shape[1])
triangle_fEM.columns = cols
triangle_fEM = triangle_fEM * aux_vigencia


🔔❗ Nuevo flujo de mortalidad (qx)

In [ ]:
triangulo_coberturaEM = mat_mortalidadEM * triangle_fEM
triangulo_coberturaEM.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_coberturaEM.columns = cols
triangulo_coberturaEM = triangulo_coberturaEM * mat_zer.iloc[:,3:]

triangulo_coberturaEM.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_coberturaEM = triangulo_coberturaEM.iloc[:,0:(max(rmat['periodo restante'])+1)]

🔔❗ Nuevo flujo de caidas (cx)

In [ ]:
triangulo_caidasEM = mat_caidas * triangle_fEM

triangulo_caidasEM.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidasEM.columns = cols

triangulo_caidasEM = triangulo_caidasEM.iloc[:, :-1] * mat_zer2.iloc[:,3:]
triangulo_caidasEM.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidasEM = triangulo_caidasEM.iloc[:,0:(max(rmat['periodo restante'])+1)]

🧮 Flujo de reservas

In [ ]:
flujo_cob1EM = beneficio * triangulo_coberturaEM.iloc[:, 1:]
flujo_gastosEM = gastos * triangle_fEM
flujo_dotalEM = dotal * ***  # el dotal se pondera con la sobrevivencia de ESTE escenario (no con la base)
flujo_primasEM = prima * triangle_fEM

egresosEM =  flujo_cob1EM + flujo_gastosEM + flujo_dotalEM
ingresosEM = flujo_primasEM
reservaEM = (egresosEM - ingresosEM) * mat_factor_descuento
sumReservaEM = reservaEM.sum(axis=1)


🧮 Reserva individual

In [ ]:
reserva_individualEM = reservaEM.sum(axis=1)
reserva_individualEM = pd.DataFrame(reserva_individualEM)
reserva_individualEM[reserva_individualEM < 0] = 0
reserva_individualEM.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individualEM.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individualEM.head()

# 📉 Riesgo de caídas
- **Estrés de un aumento del 25%**

In [ ]:
estres_caidas1 = 1 + ***  # incremento del 25%
mat_caidasEC1 = mat_caidas * estres_caidas1


🔔❗ Nuevo flujo de sobrevivencia (Px)

In [ ]:
pxEC1 =  (1 - ***.iloc[:,:-1]) * (1-mat_caidasEC1.iloc[:,:-1])  # ¿la mortalidad se estresa en este escenario de caídas?

triangle_fEC1 = pxEC1.copy()
triangle_fEC1 = triangle_fEC1.cumprod(axis=1)
triangle_fEC1 = pd.DataFrame(triangle_fEC1)
triangle_fEC1.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_fEC1.shape[1])
triangle_fEC1.columns = cols
triangle_fEC1 = triangle_fEC1 * aux_vigencia


🔔❗ Nuevo flujo de mortalidad (qx)

In [ ]:
triangulo_coberturaEC1 = qx.iloc[:,1:] * triangle_fEC1
triangulo_coberturaEC1.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_coberturaEC1.columns = cols
triangulo_coberturaEC1 = triangulo_coberturaEC1 * mat_zer.iloc[:,3:]

triangulo_coberturaEC1.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_coberturaEC1 = triangulo_coberturaEC1.iloc[:,0:(max(rmat['periodo restante'])+1)]

🔔❗ Nuevo flujo de caidas (cx)

In [ ]:
triangulo_caidasEC1 = mat_caidasEC1 * triangle_fEC1

triangulo_caidasEC1.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidasEC1.columns = cols

triangulo_caidasEC1 = triangulo_caidasEC1.iloc[:, :-1] * mat_zer2.iloc[:,3:]
triangulo_caidasEC1.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidasEC1 = triangulo_caidasEC1.iloc[:,0:(max(rmat['periodo restante'])+1)]

🧮 Flujo de reservas

In [ ]:
flujo_cob1EC1 = beneficio * triangulo_coberturaEC1.iloc[:, 1:]
flujo_gastosEC1 = gastos * triangle_fEC1
flujo_dotalEC1 = dotal * triangle_fEC1
flujo_primasEC1 = prima * triangle_fEC1

egresosEC1 =  flujo_cob1EC1 + flujo_gastosEC1 + flujo_dotalEC1
ingresosEC1 = flujo_primasEC1
reservaEC1 = (egresosEC1 - ingresosEC1) * mat_factor_descuento
sumReservaEC1 = reservaEC1.sum(axis=1)

🧮 Reserva individual

In [ ]:
reserva_individualEC1 = reservaEC1.sum(axis=1)
reserva_individualEC1 = pd.DataFrame(reserva_individualEC1)
reserva_individualEC1[reserva_individualEC1 < 0] = 0
reserva_individualEC1.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individualEC1.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individualEC1.head()

# 📉 Riesgo de caídas
- **Estrés de una disminución del 25%**

In [ ]:
estres_caidas2 = 1 - ***  # disminución del 25%
mat_caidasEC2 = mat_caidas * estres_caidas2


🔔❗ Nuevo flujo de sobrevivencia (Px)

In [ ]:
pxEC2 =  (1 - mat_mortalidad.iloc[:,:-1]) * (1-***.iloc[:,:-1])  # ¿qué matriz de caídas corresponde a ESTE escenario (-25%)?

triangle_fEC2 = pxEC2.copy()
triangle_fEC2 = triangle_fEC2.cumprod(axis=1)
triangle_fEC2 = pd.DataFrame(triangle_fEC2)
triangle_fEC2.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_fEC2.shape[1])
triangle_fEC2.columns = cols
triangle_fEC2 = triangle_fEC2 * aux_vigencia


🔔❗ Nuevo flujo de mortalidad (qx)

In [ ]:
triangulo_coberturaEC2 = qx.iloc[:,1:] * triangle_fEC2
triangulo_coberturaEC2.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_coberturaEC2.columns = cols
triangulo_coberturaEC2 = triangulo_coberturaEC2 * mat_zer.iloc[:,3:]

triangulo_coberturaEC2.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_coberturaEC2 = triangulo_coberturaEC2.iloc[:,0:(max(rmat['periodo restante'])+1)]

🔔❗ Nuevo flujo de caidas (cx)

In [ ]:
triangulo_caidasEC2 = mat_caidasEC2 * triangle_fEC2

triangulo_caidasEC2.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidasEC2.columns = cols

triangulo_caidasEC2 = triangulo_caidasEC2.iloc[:, :-1] * mat_zer2.iloc[:,3:]
triangulo_caidasEC2.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidasEC2 = triangulo_caidasEC2.iloc[:,0:(max(rmat['periodo restante'])+1)]

🧮 Flujo de reservas

In [ ]:
flujo_cob1EC2 = beneficio * triangulo_coberturaEC2.iloc[:, 1:]
flujo_gastosEC2 = gastos * triangle_fEC2
flujo_dotalEC2 = dotal * triangle_fEC2
flujo_primasEC2 = prima * triangle_fEC2

egresosEC2 =  flujo_cob1EC2 + flujo_gastosEC2 + flujo_dotalEC2
ingresosEC2 = flujo_primasEC2
reservaEC2 = (egresosEC2 - ingresosEC2) * mat_factor_descuento
sumReservaEC2 = reservaEC2.sum(axis=1)

🧮 Reserva individual

In [ ]:
reserva_individualEC2 = reservaEC2.sum(axis=1)
reserva_individualEC2 = pd.DataFrame(reserva_individualEC2)
reserva_individualEC2[reserva_individualEC2 < 0] = 0
reserva_individualEC2.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individualEC2.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individualEC2.head()

# 💡 Ejercicio

Calcula el <b>riesgo de gastos:</b>

Considerando un estrés de un <b> incremento del 10%.</b></ol> </div>

# 💸 Riesgo de gastos
- **Estrés de un aumento del 10%**

In [ ]:
gastosEG =  gastos * ***  # estrés de gastos administrativos: +10%


🧮 Flujo de reservas

In [ ]:
flujo_cob = beneficio * triangulo_cobertura.iloc[:, 1:]
flujo_gastosEG = *** * triangle_f
flujo_dotal = dotal * ***  # mismo triángulo que usaste en el escenario base
flujo_primas = prima * triangle_f

egresosEG =  flujo_cob + flujo_gastosEG + flujo_dotal
ingresos = flujo_primas

reservaEG = (egresosEG - ingresos) * ***
sumReservaEG = reservaEG.sum(axis=1)


🧮 Reserva individual

In [ ]:
reserva_individualEG = reservaEG.sum(axis=1)
reserva_individualEG = pd.DataFrame(reserva_individualEG)
reserva_individualEG[reserva_individualEG < ***] = 0  # las reservas no pueden ser negativas
reserva_individualEG.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individualEG.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individualEG.head()


In [ ]:
#-------------------------------------------------------------------------------------------------
# Valor de respaldo para la Reportería
# Si el ejercicio de "Riesgo de gastos" no fue resuelto, usamos la reserva base (sin estrés)
# como aproximación para que la Reportería no falle. En este solucionario es un no-op:
# sumReservaEG ya quedó calculada arriba.
#-------------------------------------------------------------------------------------------------
if 'sumReservaEG' not in dir():
    sumReservaEG = sumReserva.copy()


#📊 Reportería

In [ ]:
matriz_escenarios = pd.concat([sumReserva, sumReservaEM, sumReservaEC1, sumReservaEC2, sumReservaEG], axis=1, ignore_index=True)
matriz_escenarios.columns = [***, 'mortalidad', 'caidas (+)', 'caidas (-)', 'gastos']  # nombre de la columna del escenario sin estrés
matriz_escenarios[matriz_escenarios < 0] = 0

pd.options.display.float_format = '{:,.2f}'.format
pd.DataFrame(matriz_escenarios.sum(axis=0))


In [ ]:
matriz_riesgos_ =  matriz_escenarios.subtract(matriz_escenarios.iloc[:, ***], axis=0)  # columna del escenario base (posición 0 = 'modelo')
matriz_riesgos_.loc[ matriz_riesgos_['caidas (+)'] >=0, 'caidas'] = matriz_riesgos_['caidas (+)']
matriz_riesgos_.loc[ matriz_riesgos_['caidas (-)'] >=0, 'caidas'] = matriz_riesgos_['caidas (-)']
matriz_riesgos = matriz_riesgos_[['mortalidad', 'caidas', 'gastos']]

pd.options.display.float_format = '{:,.2f}'.format
pd.DataFrame(matriz_riesgos.sum(axis=0))


---

## 📎 Anexo: Factor de nivelación

El **factor de nivelación** ajusta la tabla de mortalidad `qx` a la siniestralidad observada de la cartera (por género y por año), y permite calcular un **shock de mortalidad y longevidad** (percentil 99%). No se usa dentro de esta sesión, pero es un insumo relacionado con los estreses de mortalidad que acabas de calcular en la Parte C.

Su desarrollo completo (siniestros esperados, factores por género/año, tablas ajustadas y shocks) está en **`anexos/anexo_factor_nivelacion.ipynb`** — un notebook autocontenido, ya resuelto, que puedes revisar cuando quieras profundizar en este tema.

---


# 💡 Ejercicio

Con la información brindada:
<li><b>Agrega probabilidades de caídas</b> al modelo.</li>
<li><b>Incorpora una cobertura adicional:</b> la incidencia de esta cobertura corresponde al <b>10 %</b> de la <code>qx</code> y mantiene el mismo monto de <code>suma_asegurada</code>.</li> </ol> </div>

🧠 Ejercicio 1: Incorporar caídas

> Las tablas y triángulos de caídas (`tabla2`, `triangle_c`) y la matriz `mat_zer2` **ya están construidos** en la Parte B (los reutilizamos también en la Parte C) — aquí solo falta combinarlos con `qx` para obtener la reserva con caídas.


In [ ]:
qx_matriz2 = qx.iloc[:,1:qx.shape[1]-1]
caidas = cx.iloc[:,1:cx.shape[1]-1]
px2 =  (1 - qx_matriz2) * (1 - ***)


In [ ]:
triangle_f2 = px2.copy()
triangle_f2 = triangle_f2.cumprod(axis=1)
triangle_f2 = pd.DataFrame(triangle_f2)
triangle_f2.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_f2.shape[1])
triangle_f2.columns = cols
triangle_f2 = triangle_f2 * ***

pd.options.display.float_format = '{:,.7f}'.format
triangle_f2.head()


In [ ]:
triangulo_cobertura2 = qx.iloc[:,1:qx.shape[1]] * triangle_f2.iloc[:,0:triangle_f2.shape[1]]
triangulo_cobertura2.insert(0,'cobertura1',0) # 0 al inicio
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_cobertura2.columns = cols
triangulo_cobertura2 = triangulo_cobertura2 * mat_zer.iloc[:,3:mat_zer.shape[1]]

triangulo_cobertura2.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_cobertura2 = triangulo_cobertura2.iloc[:,0:(max(rmat[***])+1)]
triangulo_cobertura2.head()


In [ ]:
flujo_cob12 = beneficio * triangulo_cobertura2.iloc[:, 1:triangulo_cobertura2.shape[1]]
flujo_gastos2 = gastos * triangle_f2
flujo_primas2 = prima * triangle_f2

egresos2 =  flujo_cob12 + flujo_gastos2
ingresos2 = flujo_primas2
reserva2 = (egresos2 - ingresos2) * ***
reserva2.head()


In [ ]:
reserva_individual2 = reserva2.sum(axis=1)
reserva_individual2 = pd.DataFrame(reserva_individual2)
reserva_individual2[reserva_individual2 < ***] = 0
reserva_individual2.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individual2.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individual2.head()


In [ ]:
resumen2 = reserva_individual2.copy()
resumen2['nombre_producto'] = rmat['nombre_producto']
#del resumen['num_poliza']
cuadro_resumen2 = resumen2.groupby(['nombre_producto']).agg({'Reservas':***})

pd.options.display.float_format = '{:,.2f}'.format
cuadro_resumen2


🧠 Ejercicio 2: Incorporar decremento complementario

🧩Parte nominal

In [ ]:
beneficio2 = ***.copy()
beneficio2.head()


🧩Parte probabilizada

In [ ]:
triangulo_cobertura3 = (qx.iloc[:,1:qx.shape[1]] * ***) * triangle_f2.iloc[:,0:triangle_f2.shape[1]]
triangulo_cobertura3.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_cobertura3.columns = cols
triangulo_cobertura3 = triangulo_cobertura3 * mat_zer.iloc[:,3:mat_zer.shape[1]]

triangulo_cobertura3.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_cobertura3 = triangulo_cobertura3.iloc[:,0:(max(rmat['periodo restante'])+1)]
triangulo_cobertura3.head()


🧮 Flujo de reservas

In [ ]:
flujo_cob2 = beneficio2 * triangulo_cobertura3.iloc[:, 1:triangulo_cobertura3.shape[1]]
flujo_gastos3 = gastos * triangle_f2
flujo_primas3 = prima * triangle_f2

egresos3 =  flujo_cob1 + *** + flujo_gastos3
ingresos3 = flujo_primas3
reserva3 = (egresos3 - ingresos3) * mat_factor_descuento
reserva3.head()


🧮 Reserva individual

In [ ]:
reserva_individual3 = reserva3.sum(axis=1)
reserva_individual3 = pd.DataFrame(reserva_individual3)
reserva_individual3[reserva_individual3 < 0] = 0
reserva_individual3.insert(0, ***, rmat['num_poliza'])
reserva_individual3.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individual3.head()


🧮 Reserva agregada

In [ ]:
resumen3 = reserva_individual3.copy()
resumen3['nombre_producto'] = rmat['nombre_producto']
cuadro_resumen3 = resumen3.groupby(['nombre_producto']).agg({'Reservas':***})

pd.options.display.float_format = '{:,.2f}'.format
cuadro_resumen3


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📌 Recap
</h1>

- Calculaste la **RRC (RPND)** de un producto de vida a partir de la base de cálculo y el porcentaje de prima no devengada.
- Construiste, **una sola vez**, el motor de reservas matemáticas: tablas de mortalidad y caídas (qx/cx), triángulos de probabilidades (con el ejemplo de 2 pólizas y luego la base real), flujos nominal / probabilizado / financiero, y la reserva individual y agregada.
- Reutilizaste ese mismo motor —sin recargar ni limpiar la base— para construir **tres escenarios de estrés** (mortalidad +11%, caídas ±25%, gastos +10%) y una **matriz de escenarios y de riesgos** que cuantifica el impacto de cada uno frente al modelo base.
- Practicaste el mismo motor en dos ejercicios de cierre: incorporar caídas como decremento adicional y una cobertura complementaria del 10% de la qx.
- Viste dónde encaja el **factor de nivelación** en este flujo (detalle completo en `anexos/anexo_factor_nivelacion.ipynb`).

**Siguiente parada:** Sesión 5 — Simulación y distribuciones de pérdida, donde vas a generar y calibrar distribuciones de frecuencia/severidad y simular pérdidas agregadas (Monte Carlo) sentando las bases del riesgo de prima que se combina, en la Sesión 6, con el riesgo de reserva que calculaste aquí.
